In [1]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [3]:
import sys
sys.path.append("..")

In [4]:
from module1.ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [5]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [6]:
len(documents)

103

In [7]:
len(ground_truth)

515

In [10]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [11]:
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
)

In [12]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [13]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [14]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, ground_truth, generate_rag_answer)

100%|██████████| 515/515 [01:57<00:00,  4.40it/s]


In [15]:
answers = []

for answer_record in results:
    answers.append(answer_record)

In [16]:
assistant.total_cost()

0.5342595000000002

In [17]:
answers[0]

{'question': 'Is it still okay to join this course if I found it late?',
 'answer_llm': 'Yes, you can still join the course even if you found it late. If you want a certificate, make sure you submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [18]:
df_answers = pd.DataFrame(answers)
df_answers.to_csv("data/rag-answers-new.csv", index=False)